# Test smiles_to_scaffold Function

Tests both scaffold (SMARTS) and removal modes, with and without scattering conditioning.


In [1]:
import torch
import numpy as np
from rdkit import Chem
import sys
import os
from pathlib import Path

# Setup paths
notebook_dir = Path().resolve()
if notebook_dir.name == 'grassy_dit':
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from grassy_dit.train import ScatteringGraphDIT
from grassy_dit.sample import smiles_to_scaffold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {project_root}")


Using device: cpu
Project root: /mnt/c/Users/stolk/OneDrive/github/GRASSY-Net


In [2]:
# Load model
scattering_moments = np.load('grassy_dit/data/scattering_moments.npy')
scattering_dim = scattering_moments.shape[1]
num_levels = 11
num_moments = 4
num_atom_types = scattering_dim // (num_levels * num_moments)

checkpoint_path = 'grassy_dit_checkpoint.pt'
ckpt = torch.load(checkpoint_path, map_location='cpu')
pos_shape = ckpt['model_state_dict']['denoiser.scatter_tokenizer.pos'].shape
checkpoint_num_tokens = pos_shape[1]
checkpoint_num_atom_types = checkpoint_num_tokens - 11

model = ScatteringGraphDIT()
model.num_atom_types = checkpoint_num_atom_types
model.num_levels = num_levels
model.num_moments = num_moments
model.load_from_local(checkpoint_path)
print(f"Model loaded: {checkpoint_num_atom_types} atom types")


Model successfully loaded from local path: grassy_dit_checkpoint.pt
Model loaded: 16 atom types


In [ ]:
# Load scattering model for computing moments
from models.LEGS_module import Scatter
from datasets.ZINCDataset import ZINCDataset, mol_to_pyg

dataset = ZINCDataset('datasets/ZINC12K.npy', prop_stat_dict='datasets/ZINC12K_stats.npy')
scatter = Scatter(in_channels=dataset.num_node_features, trainable_laziness=False).to(device)
scatter.load_state_dict(torch.load('scripts/trained_models/ZINC12K.npy', map_location=device))
scatter.eval()
print("Scattering model loaded")

def extract_scattering_from_smiles(smiles, scatter_model, device):
    """Extract scattering moments from a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Convert to PyG format
    data = mol_to_pyg(mol)
    
    # Create node features with the same encoding as ZINCDataset (16 features)
    num_node_features = 16
    node_feat = torch.zeros(data.num_nodes, num_node_features)
    
    atom_type_map = {'C': 0, 'O': 1, 'N': 2, 'S': 3, 'F': 4, 'Cl': 5, 'Br': 6, 'I': 7}
    
    for i, atom in enumerate(mol.GetAtoms()):
        entry = atom.GetSymbol()
        if entry in atom_type_map:
            node_feat[i, atom_type_map[entry]] = 1.0
        
        # Pair encoding
        neighbors = [mol.GetAtomWithIdx(nbr.GetIdx()).GetSymbol() for nbr in atom.GetNeighbors()]
        for nbr_symbol in neighbors:
            if entry == 'C' and nbr_symbol == 'O':
                node_feat[i, 8] = 1.0
            if entry == 'C' and nbr_symbol == 'N':
                node_feat[i, 9] = 1.0
            if entry == 'C' and nbr_symbol == 'S':
                node_feat[i, 10] = 1.0
            if (entry == 'O' and nbr_symbol == 'N') or (entry == 'N' and nbr_symbol == 'O'):
                node_feat[i, 11] = 1.0
            if (entry == 'O' and nbr_symbol == 'S') or (entry == 'S' and nbr_symbol == 'O'):
                node_feat[i, 12] = 1.0
            if (entry == 'N' and nbr_symbol == 'S') or (entry == 'S' and nbr_symbol == 'N'):
                node_feat[i, 13] = 1.0
            if entry == 'C' and nbr_symbol == 'F':
                node_feat[i, 14] = 1.0
            if entry == 'C' and nbr_symbol == 'Cl':
                node_feat[i, 15] = 1.0
    
    data.x = node_feat
    
    # Convert bonds to edge_attr
    edge_attr = []
    for bond in mol.GetBonds():
        bt = bond.GetBondType()
        if bt == Chem.BondType.SINGLE:
            edge_attr.append(1)
        elif bt == Chem.BondType.DOUBLE:
            edge_attr.append(2)
        elif bt == Chem.BondType.TRIPLE:
            edge_attr.append(3)
        elif bt == Chem.BondType.AROMATIC:
            edge_attr.append(3)
        else:
            edge_attr.append(1)
    
    edge_attr_full = []
    for i in range(len(edge_attr)):
        edge_attr_full.extend([edge_attr[i], edge_attr[i]])
    
    data.edge_attr = torch.tensor(edge_attr_full, dtype=torch.long)
    data = data.to(device)
    
    with torch.no_grad():
        moments, _ = scatter_model(data)
    
    return moments.cpu().numpy().flatten()

Scattering model loaded


In [4]:
# Pick a test molecule (can be any SMILES)
test_smiles = "COc1ccccc1N"  # Simple molecule with benzene ring
test_mol = Chem.MolFromSmiles(test_smiles)

# Compute scattering moments for this molecule
target_scattering = extract_scattering_from_smiles(test_smiles, scatter, device)
null_scattering = np.zeros_like(target_scattering)

print(f"Test molecule: {test_smiles}")
print(f"Number of atoms: {test_mol.GetNumAtoms()}")
print(f"Atoms: {[atom.GetSymbol() for atom in test_mol.GetAtoms()]}")
print(f"Computed scattering dimension: {len(target_scattering)}")

atom_decoder = model.dataset_info['atom_decoder']
bond_decoder = model.dataset_info.get('bond_decoder', None)
print(f"\nAtom decoder: {atom_decoder}")

Test molecule: COc1ccccc1N
Number of atoms: 9
Atoms: ['C', 'O', 'C', 'C', 'C', 'C', 'C', 'C', 'N']
Computed scattering dimension: 704

Atom decoder: ['C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'I']


In [5]:
   def test_generation(scaffold_X, scaffold_E, scaffold_mask, scattering, description, num_nodes=None, original_smiles=None):
    """Test generation and check validity/exact match."""
    num_samples = 3
    
    # Expand for batch
    scaffold_X_batch = scaffold_X.unsqueeze(0).expand(num_samples, -1, -1)
    scaffold_E_batch = scaffold_E.unsqueeze(0).expand(num_samples, -1, -1, -1)
    scaffold_mask_batch = scaffold_mask.unsqueeze(0).expand(num_samples, -1)
    
    # Generate
    generated_smiles = model.generate(
        scattering=scattering,
        num_nodes=num_nodes if num_nodes is not None else test_mol.GetNumAtoms(),
        batch_size=num_samples,
        scaffold_X=scaffold_X_batch,
        scaffold_E=scaffold_E_batch,
        scaffold_node_mask=scaffold_mask_batch,
    )
    
    # Analyze results
    target_smiles = original_smiles if original_smiles is not None else test_smiles
    original_canonical = Chem.MolToSmiles(Chem.MolFromSmiles(target_smiles))
    valid_count = 0
    exact_match_count = 0
    
    print(f"\n{description}")
    print(f"Generated molecules:")
    for i, smi in enumerate(generated_smiles):
        if smi is not None:
            mol_gen = Chem.MolFromSmiles(smi)
            if mol_gen is not None:
                valid_count += 1
                gen_canonical = Chem.MolToSmiles(mol_gen)
                is_exact = (gen_canonical == original_canonical)
                if is_exact:
                    exact_match_count += 1
                match_str = " (EXACT MATCH ✓)" if is_exact else ""
                print(f"  {i+1}. {smi}{match_str}")
            else:
                print(f"  {i+1}. {smi} (invalid SMILES)")
        else:
            print(f"  {i+1}. None")
    
    print(f"\nResults:")
    print(f"  Valid: {valid_count}/{num_samples} ({valid_count/num_samples*100:.1f}%)")
    print(f"  Exact matches: {exact_match_count}/{num_samples} ({exact_match_count/num_samples*100:.1f}%)")
    
    return valid_count, exact_match_count


## Test 1: Scaffold Mode (SMARTS Pattern) - WITH Conditioning


In [6]:
# Test scaffold mode: keep benzene ring (SMARTS pattern)
scaffold_pattern = "c1ccccc1"  # Benzene ring
scaffold_X, scaffold_E, scaffold_mask, node_mask, n_atoms = smiles_to_scaffold(
    test_smiles,
    model.max_node,
    atom_decoder,
    bond_decoder,
    scaffold_pattern=scaffold_pattern
)

scaffold_count = scaffold_mask.sum().item()
print(f"Full molecule: {n_atoms} atoms")
print(f"Scaffold (fixed): {scaffold_count} atoms")
print(f"Pattern: {scaffold_pattern}")

valid1, exact1 = test_generation(
    scaffold_X, scaffold_E, scaffold_mask, 
    target_scattering,
    "Scaffold Mode (SMARTS) - WITH scattering conditioning"
)


Full molecule: 9 atoms
Scaffold (fixed): 6 atoms
Pattern: c1ccccc1

Scaffold Mode (SMARTS) - WITH scattering conditioning
Generated molecules:
  1. O.O=cc1ccccc1 (invalid SMILES)
  2. O.O=Cc1ccccc1
  3. N.O=[O+]c1ccccc1

Results:
  Valid: 2/3 (66.7%)
  Exact matches: 0/3 (0.0%)


## Test 2: Scaffold Mode (SMARTS Pattern) - WITHOUT Conditioning


In [7]:
# Same scaffold, but without scattering conditioning
valid2, exact2 = test_generation(
    scaffold_X, scaffold_E, scaffold_mask,
    null_scattering,
    "Scaffold Mode (SMARTS) - WITHOUT scattering conditioning"
)



Scaffold Mode (SMARTS) - WITHOUT scattering conditioning
Generated molecules:
  1. N#Cc1ccccc1O
  2. C#[N+]c1ccccc1S
  3. N#Cc1ccccc1.O

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 0/3 (0.0%)


## Test 3: Removal Mode - WITH Conditioning


In [8]:
# Test removal mode: remove first 2 atoms (indices 0, 1)
remove_indices = [0]
scaffold_X2, scaffold_E2, scaffold_mask2, node_mask2, n_atoms2 = smiles_to_scaffold(
    test_smiles,
    model.max_node,
    atom_decoder,
    bond_decoder,
    remove_indices=remove_indices
)

scaffold_count2 = scaffold_mask2.sum().item()
print(f"Full molecule: {n_atoms2} atoms")
print(f"Scaffold (fixed): {scaffold_count2} atoms")
print(f"Removed indices: {remove_indices}")

valid3, exact3 = test_generation(
    scaffold_X2, scaffold_E2, scaffold_mask2,
    target_scattering,
    "Removal Mode - WITH scattering conditioning"
)


Full molecule: 9 atoms
Scaffold (fixed): 8 atoms
Removed indices: [0]

Removal Mode - WITH scattering conditioning
Generated molecules:
  1. Nc1ccccc1O.O
  2. Nc1ccccc1O.O
  3. Nc1ccccc1[O+]=O

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 0/3 (0.0%)


## Test 4: Removal Mode - WITHOUT Conditioning


In [9]:
# Same removal, but without scattering conditioning
valid4, exact4 = test_generation(
    scaffold_X2, scaffold_E2, scaffold_mask2,
    null_scattering,
    "Removal Mode - WITHOUT scattering conditioning"
)



Removal Mode - WITHOUT scattering conditioning
Generated molecules:
  1. NOc1ccccc1N
  2. N=[O+]c1ccccc1N
  3. COc1ccccc1N (EXACT MATCH ✓)

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 1/3 (33.3%)


## Summary


In [10]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"\n1. Scaffold (SMARTS) + Conditioning:")
print(f"   Valid: {valid1}/3, Exact: {exact1}/3")
print(f"\n2. Scaffold (SMARTS) + No Conditioning:")
print(f"   Valid: {valid2}/3, Exact: {exact2}/3")
print(f"\n3. Removal + Conditioning:")
print(f"   Valid: {valid3}/3, Exact: {exact3}/3")
print(f"\n4. Removal + No Conditioning:")
print(f"   Valid: {valid4}/3, Exact: {exact4}/3")
print("\n" + "="*60)


SUMMARY

1. Scaffold (SMARTS) + Conditioning:
   Valid: 2/3, Exact: 0/3

2. Scaffold (SMARTS) + No Conditioning:
   Valid: 3/3, Exact: 0/3

3. Removal + Conditioning:
   Valid: 3/3, Exact: 0/3

4. Removal + No Conditioning:
   Valid: 3/3, Exact: 1/3



In [ ]:
# Test removal mode: remove atoms 1, 2, 3 from molecule 0 in sequence
from datasets.load_ZINC_tranche import ZINCDataset

dataset = ZINCDataset('datasets/ZINC12K.npy', prop_stat_dict='datasets/ZINC12K_stats.npy')
molecule_idx = 0
original_smiles = dataset.smi[molecule_idx]
original_scattering = scattering_moments[molecule_idx]
original_mol = Chem.MolFromSmiles(original_smiles)
original_n_atoms = original_mol.GetNumAtoms()

print(f"Original molecule: {original_smiles}")
print(f"Original molecule has {original_n_atoms} atoms")

# Test removing 1, 2, 3 atoms in sequence
results_removal_conditioned = []

for num_removals in [1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"REMOVING {num_removals} ATOM(S) - WITH Conditioning")
    print(f"{'='*60}")
    
    # Create removal list: remove first N atoms (indices 0, 1, 2, ...)
    remove_indices = list(range(num_removals))
    
    scaffold_X, scaffold_E, scaffold_mask, node_mask, n_atoms = smiles_to_scaffold(
        original_smiles,
        model.max_node,
        atom_decoder,
        bond_decoder,
        remove_indices=remove_indices
    )
    
    scaffold_count = scaffold_mask.sum().item()
    print(f"Full molecule: {n_atoms} atoms")
    print(f"Scaffold (fixed): {scaffold_count} atoms")
    print(f"Removed indices: {remove_indices}")

    valid, exact = test_generation(
        scaffold_X, scaffold_E, scaffold_mask,
        original_scattering,  # Use matching scattering
        f"Removal Mode ({num_removals} atoms) - WITH scattering conditioning",
        num_nodes=original_n_atoms,
        original_smiles=original_smiles
    )
    
    results_removal_conditioned.append({
        'num_removals': num_removals,
        'valid': valid,
        'exact': exact
    })

Original molecule: COC1=CC=C2C=CC(O)=C(CN3CCN(S(=O)(=O)C4=CN(C)C=C4)CC3)C2=C1
Original molecule has 29 atoms

REMOVING 1 ATOM(S) - WITH Conditioning
Full molecule: 29 atoms
Scaffold (fixed): 28 atoms
Removed indices: [0]

Removal Mode (1 atoms) - WITH scattering conditioning
Generated molecules:
  1. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  2. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  3. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 3/3 (100.0%)

REMOVING 2 ATOM(S) - WITH Conditioning
Full molecule: 29 atoms
Scaffold (fixed): 27 atoms
Removed indices: [0, 1]

Removal Mode (2 atoms) - WITH scattering conditioning
Generated molecules:
  1. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  2. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  3. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)

Results:
  

In [12]:
# Same removal tests but WITHOUT scattering conditioning
print(f"\n{'='*60}")
print("REMOVAL MODE - WITHOUT Conditioning")
print(f"{'='*60}")

results_removal_unconditioned = []

for num_removals in [1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"REMOVING {num_removals} ATOM(S) - WITHOUT Conditioning")
    print(f"{'='*60}")
    
    # Create removal list: remove first N atoms (indices 0, 1, 2, ...)
    remove_indices = list(range(num_removals))
    
    scaffold_X, scaffold_E, scaffold_mask, node_mask, n_atoms = smiles_to_scaffold(
        original_smiles,
        model.max_node,
        atom_decoder,
        bond_decoder,
        remove_indices=remove_indices
    )
    
    scaffold_count = scaffold_mask.sum().item()
    print(f"Full molecule: {n_atoms} atoms")
    print(f"Scaffold (fixed): {scaffold_count} atoms")
    print(f"Removed indices: {remove_indices}")
    
    valid, exact = test_generation(
        scaffold_X, scaffold_E, scaffold_mask,
        null_scattering,  # No conditioning
        f"Removal Mode ({num_removals} atoms) - WITHOUT scattering conditioning",
        num_nodes=original_n_atoms,
        original_smiles=original_smiles
    )
    
    results_removal_unconditioned.append({
        'num_removals': num_removals,
        'valid': valid,
        'exact': exact
    })


REMOVAL MODE - WITHOUT Conditioning

REMOVING 1 ATOM(S) - WITHOUT Conditioning
Full molecule: 29 atoms
Scaffold (fixed): 28 atoms
Removed indices: [0]

Removal Mode (1 atoms) - WITHOUT scattering conditioning
Generated molecules:
  1. Cn1ccc(S(=O)(=O)N2CCN(Cc3c(O)ccc4ccc([O+]=N)cc34)CC2)c1
  2. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  3. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 2/3 (66.7%)

REMOVING 2 ATOM(S) - WITHOUT Conditioning
Full molecule: 29 atoms
Scaffold (fixed): 27 atoms
Removed indices: [0, 1]

Removal Mode (2 atoms) - WITHOUT scattering conditioning
Generated molecules:
  1. COc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1 (EXACT MATCH ✓)
  2. C=Cc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1
  3. CCc1ccc2ccc(O)c(CN3CCN(S(=O)(=O)c4ccn(C)c4)CC3)c2c1

Results:
  Valid: 3/3 (100.0%)
  Exact matches: 1/3 (33.3%)

REMOVING 3 ATOM(S) - WITHOUT Conditioning
Full molecul